# **Ingestion pipeline**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# Data -> Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

/tmp/ipykernel_1872/2830746044.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyPDFLoader


In [12]:
def load_all_pdfs():
  folder_path = "/content/drive/MyDrive/Data1"
  num_docs = 0
  all_docs = []

  for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
      # complete file path
      pdf_path = os.path.join(folder_path, filename)

      loader = PyPDFLoader(pdf_path)
      doc = loader.load()

      all_docs.extend(doc)
      num_docs += 1

  print("total pdfs:", num_docs)
  print("total pages:", len(all_docs))
  return all_docs

In [13]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 40


In [15]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

# ***CHUNKS***

In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size = 500, chunk_overlap=50):

  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size = chunk_size,
      chunk_overlap = chunk_overlap
  )

  chunked_docs = text_splitter.split_documents(documents)
  return chunked_docs

In [17]:
chunks = split_docs(all_pdf_documents)

In [18]:
len(chunks)

402

In [19]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hyperref', 'creationdate': '2021-04-13T00:48:38+00:00', 'author': '', 'keywords': '', 'moddate': '2021-04-13T00:48:38+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/content/drive/MyDrive/Data1/research1.pdf', 'total_pages': 19, 'page': 0, 'page_label': '1'}, page_content='Retrieval-Augmented Generation for\nKnowledge-Intensive NLP Tasks\nPatrick Lewis†‡, Ethan Perez⋆,\nAleksandra Piktus†, Fabio Petroni†, Vladimir Karpukhin†, Naman Goyal†, Heinrich Küttler†,\nMike Lewis†, Wen-tau Yih†, Tim Rocktäschel†‡, Sebastian Riedel†‡, Douwe Kiela†\n†Facebook AI Research;‡University College London;⋆New York University;\nplewis@fb.com\nAbstract\nLarge pre-trained language models have been shown to store factual knowledge'),
 Document(metadata={'producer': 'pdfTeX-1.40.21', 'creator': 'LaTeX with hype

# **Embedding**

In [20]:
from sentence_transformers import SentenceTransformer

In [37]:
class EmbeddingManager:
  def __init__(self, model_name="all-MiniLM-L6-v2"):


    self.model_name=model_name
    print("loading model...", self.model_name)
    self.model = SentenceTransformer(self.model_name)
    print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


  def generate_embeddings(self, text):
    embeddings = self.model.encode(text , show_progress_bar=True)
    print("embedding shape", embeddings.shape)
    return embeddings

In [38]:
embedding_manager = EmbeddingManager()

loading model... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


/tmp/ipykernel_1872/3343746223.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


# **Vector Store**

In [39]:
import chromadb
import uuid

In [48]:
class VectorStoreManager:
  def __init__(self, persist_directory="/content/drive/MyDrive/Data1/vector_store", collection_name="pdf_documents"):
    self.collection_name = collection_name
    self.persist_directory = persist_directory
    self.collection = None
    self.client = None


    self._initialize_store()

  def _initialize_store(self):
    os.makedirs(self.persist_directory, exist_ok=True)

    # Create a client
    self.client = chromadb.PersistentClient(path=self.persist_directory)

    # Create a collection
    self.collection = self.client.get_or_create_collection(
      name=self.collection_name,
      metadata={"description": "vector store collection for pdf embeddings in RAG"}
    )

    print("initialized the vector store with collection:", self.collection_name)
    print("docs in collection:", self.collection.count())


  def add_documents(self, documents, embeddings):
    if len(documents) != len(embeddings):
        raise ValueError("num of documents does not match num of embeddings")

    ids = []
    all_metadata = []
    documents_content = []
    embeddings_list = []

    for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
        doc_id = f"doc_{uuid.uuid4()}"
        ids.append(doc_id)
        metadata = dict(doc.metadata)
        metadata["doc_index"] = i
        metadata["content_length"] = len(doc.page_content)
        all_metadata.append(metadata)
        documents_content.append(doc.page_content)
        embeddings_list.append(embedding.tolist())

    # Un-indent this block so it sits outside the for loop
    self.collection.add(
        ids=ids,
        metadatas=all_metadata,  # Note: ChromaDB uses 'metadatas' (plural)
        documents=documents_content,
        embeddings=embeddings_list
    )

    print("total documents added in vector store=", len(documents_content))
    print("docs in collection:", self.collection.count())

In [49]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [50]:
# data -> documents -> chunks -> embedding -> store in vector database
texts = [doc.page_content for doc in chunks]

embedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

embedding shape (402, 384)
total documents added in vector store= 402
docs in collection: 402


# **Retrieval pipeline**

In [51]:
from sklearn.metrics.pairwise import cosine_similarity

In [57]:
class RAGRetriever:
  def __init__(self, embedding_manager, vector_store):
    self.embedding_manager = embedding_manager
    self.vector_store = vector_store


  def retrieve(self, query, top_k=5, score_threshold=0.0):
    # query => embedding
    query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

    # semantic search
    results = self.vector_store.collection.query(
        query_embeddings=[query_embeddings.tolist()],
        n_results=top_k
    )

    # process results
    retrieved_docs = []
    if results["documents"] and results["documents"][0]:
        ids = results["ids"][0]
        metadatas = results["metadatas"][0]  # Fixed typo ('metadatas')
        documents = results["documents"][0]
        distances = results["distances"][0]

        for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
            similarity_score = 1 - distance
            if similarity_score >= score_threshold:
                retrieved_docs.append({
                    "id": doc_id,
                    "document": document,
                    "metadata": metadata,
                    "distance": distance,
                    "similarity_score": similarity_score,
                    "rank": i + 1
                })
        print(f"retrieved {len(retrieved_docs)} documents")
    else:
        print("no documents found")

    return retrieved_docs

In [58]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [59]:
rag_retriever.retrieve("What is RAG?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape (1, 384)
retrieved 5 documents


[{'id': 'doc_fddfac34-afc3-430c-8061-7173cc8151a8',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'page': 0,
   'producer': 'pdfTeX-1.40.25',
   'moddate': '2024-03-28T00:54:45+00:00',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'subject': '',
   'author': '',
   'content_length': 288,
   'creationdate': '2024-03-28T00:54:45+00:00',
   'page_label': '1',
   'source': '/content/drive/MyDrive/Data1/research2.pdf',
   'doc_index': 169,
   'total_pages': 21,
   'creator': 'LaTeX with hyperref',
   'keywords': '',
   'title': '',
   'trapped': '/False'},
  'distance': 0.46291500329971313,
  'similarity_score': 0.5370849967002869,
  'rank': 1},
 {

## **Integrate with LLMs**

# **GROQ**

In [106]:
from groq import Groq

client = Groq(api_key=API_KEY_GROQ)

models = client.models.list()

for model in models.data:
    print(model.id)

canopylabs/orpheus-arabic-saudi
openai/gpt-oss-safeguard-20b
whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-86m
qwen/qwen3.8-27b
whisper-large-v3
canopylabs/orpheus-v1-english
openai/gpt-oss-20b
allam-2-7b
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-22m


In [ ]:
import os
API_KEY_GROQ = os.environ["GROQ_API_KEY"]

In [108]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=API_KEY_GROQ,
    model="openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024
)

In [109]:
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)
    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("We found no relevant context for the given query.")

    prompt = f"""Use the given context to generate the answer for the query:
                    Context:{context}
                    Query:{query}"""

    response = llm.invoke([prompt])
    return response.content

In [115]:
answer = generate_output("what is ingestion and retreval?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embedding shape (1, 384)
retrieved 0 documents
We found no relevant context for the given query.


In [116]:
print(answer)

**Ingestion** and **retrieval** are two fundamental stages in any data‑centric system, especially in data engineering, analytics, and information‑retrieval contexts.

---

## 1. Data Ingestion

| Aspect | What it is | Why it matters |
|--------|------------|----------------|
| **Definition** | The process of collecting, importing, and storing data from one or more source systems into a target system (e.g., a data lake, warehouse, or search index). | It turns raw, disparate data into a usable format for downstream analytics, machine learning, or reporting. |
| **Typical Sources** | • Databases (SQL, NoSQL) <br>• APIs & web services <br>• IoT devices & sensors <br>• Log files & event streams <br>• Third‑party data feeds |
| **Common Methods** | • **Batch ingestion** – scheduled pulls (e.g., nightly ETL jobs). <br>• **Streaming ingestion** – real‑time capture (e.g., Kafka, Flink, Spark Structured Streaming). <br>• **Change Data Capture (CDC)** – capturing only changes to source data. |
| 